In [1]:
#start

In [2]:
# =========================================================
# Nemotron LoRA SFT - Kaggle fixed version (final)
# 修复内容：
# 1) cutlass 路径
# 2) ptxas-blackwell 可执行权限
# 3) dtype 替代 torch_dtype
# 4) eval_strategy / evaluation_strategy 参数兼容
# 5) 只对答案部分计算 loss
# 6) 修复 notebook 环境下 trainer.evaluate() 回调报错
# =========================================================

import os
import sys
import site
import glob
import stat
import shutil
import subprocess
import importlib.util

# ==========================================
# 0. 必须先做环境修复（在 transformers 之前）
# ==========================================
print("Setting up environment...")

utility_paths = glob.glob("/kaggle/usr/lib/notebooks/*/nvidia*utility*script")
if not utility_paths:
    raise RuntimeError("Could not find nvidia utility script path")

utility_dir = utility_paths[0]
print("utility_dir =", utility_dir)

# ---- cutlass path ----
cutlass_pkg_path = os.path.join(utility_dir, "nvidia_cutlass_dsl", "python_packages")
if not os.path.isdir(cutlass_pkg_path):
    raise RuntimeError(f"cutlass path not found: {cutlass_pkg_path}")

if cutlass_pkg_path not in sys.path:
    sys.path.insert(0, cutlass_pkg_path)
site.addsitedir(cutlass_pkg_path)

print("cutlass spec:", importlib.util.find_spec("cutlass"))
import cutlass
print("cutlass imported from:", cutlass.__file__)

# ---- Triton ptxas binaries: copy to writable dir and chmod +x ----
readonly_bin_dir = os.path.join(utility_dir, "triton", "backends", "nvidia", "bin")
writable_bin_dir = "/kaggle/working/triton_bin"
os.makedirs(writable_bin_dir, exist_ok=True)

def copy_executable(src_name, dst_name=None):
    dst_name = dst_name or src_name
    src = os.path.join(readonly_bin_dir, src_name)
    dst = os.path.join(writable_bin_dir, dst_name)

    if not os.path.exists(src):
        print(f"[WARN] missing source binary: {src}")
        return None

    shutil.copy2(src, dst)
    mode = os.stat(dst).st_mode
    os.chmod(
        dst,
        mode
        | stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR
        | stat.S_IXGRP | stat.S_IXOTH
    )
    print(f"Copied {src_name} -> {dst}")
    print("Executable:", os.access(dst, os.X_OK))
    return dst

ptxas_blackwell_path = copy_executable("ptxas-blackwell")
ptxas_path = copy_executable("ptxas")

if ptxas_blackwell_path is None:
    raise RuntimeError("Failed to prepare writable ptxas-blackwell")

if ptxas_path is None:
    ptxas_path = ptxas_blackwell_path

os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = ptxas_blackwell_path
os.environ["TRITON_PTXAS_PATH"] = ptxas_path

print("TRITON_PTXAS_BLACKWELL_PATH =", os.environ["TRITON_PTXAS_BLACKWELL_PATH"])
print("TRITON_PTXAS_PATH           =", os.environ["TRITON_PTXAS_PATH"])

print(subprocess.check_output([ptxas_blackwell_path, "--version"]).decode("utf-8").splitlines()[0])
print(subprocess.check_output([ptxas_path, "--version"]).decode("utf-8").splitlines()[0])

# ==========================================
# 1. 导入训练依赖
# ==========================================
print("Importing libraries...")

import torch
import polars as pl
import kagglehub

from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
)

# Notebook callback 修复用
try:
    from transformers.utils.notebook import NotebookProgressCallback
except Exception:
    NotebookProgressCallback = None

# ==========================================
# 2. 配置
# ==========================================
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32

MAX_LENGTH = 512          # 快速测试先 512；正式训练可改 1024
MAX_STEPS = 25           # 快速测试先 50；正式训练可改 -1
NUM_EPOCHS = 1
LEARNING_RATE = 5e-5
GRAD_ACCUM_STEPS = 16
WARMUP_RATIO = 0.03
LOGGING_STEPS = 10
EVAL_STEPS = 500
SAVE_STEPS = 500
ENABLE_THINKING = False

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
TORCH_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print(f"USE_BF16: {USE_BF16}")
print(f"TORCH_DTYPE: {TORCH_DTYPE}")

# ==========================================
# 3. 读取 train.csv
# ==========================================
print("Searching for train.csv...")
train_csv_path = None
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if filename == "train.csv":
            train_csv_path = os.path.join(dirname, filename)
            break
    if train_csv_path:
        break

if not train_csv_path:
    raise FileNotFoundError("Could not find train.csv in /kaggle/input/")

print(f"Loading data from: {train_csv_path}")
train_df = pl.read_csv(train_csv_path)

required_cols = {"prompt", "answer"}
missing_cols = required_cols - set(train_df.columns)
if missing_cols:
    raise ValueError(f"train.csv is missing required columns: {missing_cols}")

dataset = Dataset.from_pandas(train_df.to_pandas(), preserve_index=False)
dataset = dataset.shuffle(seed=42)

eval_size = min(200, max(50, int(0.02 * len(dataset))))
if eval_size >= len(dataset):
    eval_size = max(1, len(dataset) // 10)

split_dataset = dataset.train_test_split(test_size=eval_size, seed=42)
raw_train_dataset = split_dataset["train"]
raw_eval_dataset = split_dataset["test"]

print(f"Train size: {len(raw_train_dataset)}")
print(f"Eval size : {len(raw_eval_dataset)}")

# ==========================================
# 4. 加载 tokenizer / model
# ==========================================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=TORCH_DTYPE,
)

model.config.use_cache = False
print("Model loaded successfully!")

# ==========================================
# 5. 数据预处理
#    - 官方 chat template
#    - 只对答案部分算 loss
#    - 自动 boxed
# ==========================================
def normalize_answer(answer: str) -> str:
    answer = str(answer).strip()
    if "\\boxed{" in answer:
        return answer
    return f"\\boxed{{{answer}}}"

def render_prompt(prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def preprocess_example(example):
    prompt = str(example["prompt"]).strip()
    answer = normalize_answer(example["answer"])

    prompt_text = render_prompt(prompt)
    full_text = prompt_text + answer + tokenizer.eos_token

    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    labels = input_ids.copy()

    prompt_len = min(len(prompt_tokens["input_ids"]), len(labels))
    labels[:prompt_len] = [-100] * prompt_len

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

print("Tokenizing train dataset...")
train_dataset = raw_train_dataset.map(
    preprocess_example,
    remove_columns=raw_train_dataset.column_names,
)

print("Tokenizing eval dataset...")
eval_dataset = raw_eval_dataset.map(
    preprocess_example,
    remove_columns=raw_eval_dataset.column_names,
)

# ==========================================
# 6. LoRA
# ==========================================
print("Initializing LoRA adapter...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "in_proj",
        "out_proj",
        "up_proj",
        "down_proj",
        "gate_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
else:
    def make_inputs_require_grad(module, inputs, output):
        output.requires_grad_(True)
    model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

# ==========================================
# 7. TrainingArguments
# ==========================================
training_kwargs = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    optim="adamw_torch",
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
)

if MAX_STEPS is not None and MAX_STEPS > 0:
    training_kwargs["max_steps"] = MAX_STEPS

# 兼容不同 transformers 版本
try:
    training_args = TrainingArguments(**training_kwargs)
except TypeError as e:
    if "eval_strategy" in str(e):
        training_kwargs["evaluation_strategy"] = training_kwargs.pop("eval_strategy")
        training_args = TrainingArguments(**training_kwargs)
    else:
        raise

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)




Setting up environment...
utility_dir = /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script
cutlass spec: ModuleSpec(name='cutlass', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7cf01d969c10>, origin='/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages/cutlass/__init__.py', submodule_search_locations=['/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages/cutlass'])
cutlass imported from: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages/cutlass/__init__.py
Copied ptxas-blackwell -> /kaggle/working/triton_bin/ptxas-blackwell
Executable: True
Copied ptxas -> /kaggle/working/triton_bin/ptxas
Executable: True
TRITON_PTXAS_BLACKWELL_PATH = /kaggle/working/triton_bin/ptxas-blackwell
TRITON_PTXAS_PATH           = /kaggle/working/triton_bin/ptxas
ptxas-blackwell: NVIDIA (R) Ptx optimizing assembler
ptxas: NVIDIA (R) Ptx optimizing asse

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


USE_BF16: True
TORCH_DTYPE: torch.bfloat16
Searching for train.csv...
Loading data from: /kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv
Train size: 9310
Eval size : 190
Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded successfully!
Tokenizing train dataset...


...

Tokenizing eval dataset...


Map:   0%|          | 0/190 [00:00<?, ? examples/s]

Initializing LoRA adapter...


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228


In [3]:
# ==========================================
# 8. 开始训练
# ==========================================
print("Starting training...")
train_result = trainer.train()
print("Training finished.")

# 修复 notebook/kaggle 环境下最终 evaluate 的 callback 报错
if NotebookProgressCallback is not None:
    try:
        trainer.remove_callback(NotebookProgressCallback)
        print("NotebookProgressCallback removed before final evaluation.")
    except Exception as e:
        print(f"[WARN] Failed to remove NotebookProgressCallback: {e}")

print("Running final evaluation...")
eval_metrics = trainer.evaluate()
print("Eval metrics:", eval_metrics)


Starting training...


Step,Training Loss,Validation Loss


Training finished.
NotebookProgressCallback removed before final evaluation.
Running final evaluation...
Eval metrics: {'eval_loss': 0.6890793442726135, 'eval_runtime': 126.9772, 'eval_samples_per_second': 1.496, 'eval_steps_per_second': 1.496, 'epoch': 0.04296455424274973}


In [4]:
# ==========================================
# 9. 保存 adapter
# ==========================================
print(f"Saving adapter to {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

adapter_model_path = os.path.join(OUTPUT_DIR, "adapter_model.safetensors")
adapter_config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
submission_zip_path = os.path.join(OUTPUT_DIR, "submission.zip")

if not os.path.exists(adapter_model_path):
    raise FileNotFoundError(f"Missing adapter file: {adapter_model_path}")
if not os.path.exists(adapter_config_path):
    raise FileNotFoundError(f"Missing adapter config: {adapter_config_path}")

print("Creating submission.zip...")
subprocess.run(
    ["zip", "-j", submission_zip_path, adapter_model_path, adapter_config_path],
    check=True,
)

print("All done!")
print(f"Submission file created at: {submission_zip_path}")

Saving adapter to /kaggle/working...
Creating submission.zip...
  adding: adapter_model.safetensors (deflated 13%)
  adding: adapter_config.jsonAll done!
Submission file created at: /kaggle/working/submission.zip
 (deflated 58%)
